In [2]:
#skip this cell.
#This was for library development purposes only.

import sys
from pathlib import Path

src_path = Path("src")

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


In [3]:
import PyVisualFields
print(PyVisualFields.__file__)
print(PyVisualFields.__version__)

/home/eslamim/Partners HealthCare Dropbox/Mohammad Eslami/GithubRepos/PyVisualField/src/PyVisualFields/__init__.py
2.0.4


## This notebook shows examples of doing normalization and compute deviations and probabilities

### getnv(): get current normative setting/environment (from visualFields package)

In [4]:
from PyVisualFields import visualFields

######## Get the current normalization information
currentNV = visualFields.getnv()

$name
[1] "24-2"

$perimetry
[1] "static automated perimetry"

$strategy
[1] "SITA Standard"

$size
[1] "Size III"


### Compute each of the td, pd, pdp, etc based on the current NV 
### Alias to the getallvalues
### Pay attention to the inputs

# Example 
 always use canonicalize_vf_df() to make data format compatible

In [5]:
# sometimes we may have partial dataset. for example UWHVF has senstitivity, TD and PD values.
# in that case
import pandas as pd
from PyVisualFields.utils import canonicalize_vf_df, canonicalize_vf_row
from PyVisualFields.utils import vf_blocks, missing_blocks
from PyVisualFields.utils import compute_missing_blocks
from PyVisualFields.utils import print_vf_summary, investigate_vf_df

url = "https://raw.githubusercontent.com/uw-biomedical-ml/uwhvf/master/CSV/VF_Data.csv"

df_UWHVF = pd.read_csv(url)
df_UWHVF.head()

df_VFs_py = canonicalize_vf_df(df_UWHVF)

print('---> blocks',vf_blocks(df_VFs_py))
print('---> missed blocks', missing_blocks(df_VFs_py))

#### now you can investigate the available information in the dataframe
print_vf_summary(df_VFs_py) # this function will print

info = investigate_vf_df(df_VFs_py)# this function will extract the information only


---> blocks {'sens': True, 'td': True, 'pd': True, 'tdp': False, 'pdp': False}
---> missed blocks ['tdp', 'pdp']
=== VF Data Summary ===
Rows: 28943
Columns: 184

Pointwise blocks:
  ✓ sensitivity (54 locations)
  ✓ total_deviation (54 locations)
  ✓ pattern_deviation (54 locations)
  ✗ total_deviation_probability (0 locations)
  ✗ pattern_deviation_probability (0 locations)

Global indices:
  ✓ md
  ✓ psd
  ✗ ght
  ✗ vfi
  ✓ msens
  ✗ ssens
  ✗ tmd
  ✗ tsd
  ✗ pmd
  ✓ gh

Metadata:
  ✓ patientid
  ✓ eyeid
  ✗ date
  ✓ age
  ✗ yearsfollowed
  ✗ duration


In [6]:
# In above cases you can manually extract the missed values using 
# use following function to fill the missed blocks

from PyVisualFields.utils import compute_missing_blocks

df_VFs = compute_missing_blocks(df_VFs_py)
print(df_VFs_py.shape)
print('---> blocks',vf_blocks(df_VFs))
print('---> missed blocks', missing_blocks(df_VFs))
print(df_VFs.shape)

(28943, 184)
---> blocks {'sens': True, 'td': True, 'pd': True, 'tdp': True, 'pdp': True}
---> missed blocks []
(28943, 292)


In [7]:

# in the case you don't have missing blocks, but missing values, 
# You can compute the values step by step 
# you can also use them row-wise

from PyVisualFields import visualFields

df_VFs_py = canonicalize_vf_df(visualFields.data_vfpwgRetest24d2())
df_td = visualFields.gettd(df_VFs_py) # get TD values
df_tdp = visualFields.gettdp(df_td) # get TD probability values
df_pd = visualFields.getpd(df_td) # get PD values
df_pdp = visualFields.getpdp(df_pd) # get PD probability values
gh = visualFields.getgh(df_td) # get the general height
df_pdp.head()


,patientid,eyeid,date,time,age,type,fpr,fnr,fl,duration,...,pdp45,pdp46,pdp47,pdp48,pdp49,pdp50,pdp51,pdp52,pdp53,pdp54
0,1,OD,2008-08-13,00:00:00,53,pwg,0,0.0,0.00,00:00:00,...,0.95,0.95,0.95,0.95,0.950,0.95,0.95,0.95,0.95,0.99
1,1,OD,2008-08-20,00:00:00,53,pwg,0,0.0,0.06,00:00:00,...,0.95,0.95,0.95,0.95,0.950,0.95,0.95,0.95,0.95,0.95
2,1,OD,2008-08-27,00:00:00,53,pwg,0,0.0,0.00,00:00:00,...,0.95,0.95,0.95,0.05,0.005,0.95,0.95,0.95,0.95,0.95
3,1,OD,2008-09-03,00:00:00,53,pwg,0,0.0,0.06,00:00:00,...,0.95,0.95,0.95,0.05,0.010,0.95,0.95,0.95,0.95,0.95
4,1,OD,2008-09-10,00:00:00,53,pwg,0,0.0,0.13,00:00:00,...,0.95,0.95,0.95,0.02,0.005,0.95,0.95,0.95,0.95,0.95


In [8]:
from PyVisualFields.utils import canonicalize_vf_df, canonicalize_vf_row
from PyVisualFields.utils import vf_blocks, missing_blocks
from PyVisualFields.utils import compute_missing_blocks
from PyVisualFields.utils import print_vf_summary, investigate_vf_df

df_VFs_py = canonicalize_vf_df(visualFields.data_vfpwgRetest24d2())

df_VFs = compute_missing_blocks(df_VFs_py)

df_gi = visualFields.getgl(df_VFs) # get global indices
""" vfi, msens (MS), ssens, tmd, tsd, pmd, psd, gh, """
""" Notice that it will calculate only missed indexes."""
df_gi.head()

############# combine them later if you need:
#combined_df = pd.concat([df_VFs, df_gi], axis=1)
#final_df = combined_df.loc[:, ~combined_df.columns.duplicated()]


==> py_getgl: missing global indices to compute: ['msens', 'ssens', 'tmd', 'tsd', 'pmd', 'psd', 'gh', 'vfi']


,patientid,eyeid,date,time,age,type,fpr,fnr,fl,duration,msens,ssens,tmd,tsd,pmd,psd,gh,vfi
0,1,OD,2008-08-13,00:00:00,53,pwg,0,0.0,0.00,00:00:00,24.288462,6.800624,-6.110470,6.614332,-4.326164,6.644596,-1.865947,88.558160
1,1,OD,2008-08-20,00:00:00,53,pwg,0,0.0,0.06,00:00:00,26.500000,6.708204,-4.033858,6.833541,-4.590968,6.881649,0.481116,90.805091
2,1,OD,2008-08-27,00:00:00,53,pwg,0,0.0,0.00,00:00:00,24.615385,4.516436,-5.949572,4.303623,-4.111716,4.293826,-1.865947,89.054985
3,1,OD,2008-09-03,00:00:00,53,pwg,0,0.0,0.06,00:00:00,25.269231,4.182669,-5.250323,4.037272,-3.579162,4.052251,-1.700423,91.521484
4,1,OD,2008-09-10,00:00:00,53,pwg,0,0.0,0.13,00:00:00,25.596154,3.471224,-4.920857,3.146895,-3.348326,3.129952,-1.581507,93.236637


In [9]:
from PyVisualFields.utils import canonicalize_vf_df, canonicalize_vf_row
from PyVisualFields.utils import vf_blocks, missing_blocks
from PyVisualFields.utils import compute_missing_blocks
from PyVisualFields.utils import print_vf_summary, investigate_vf_df

url = "https://raw.githubusercontent.com/uw-biomedical-ml/uwhvf/master/CSV/VF_Data.csv"

df_UWHVF = pd.read_csv(url)
df_UWHVF.head()

df_VFs_py = canonicalize_vf_df(df_UWHVF)
df_VFs = compute_missing_blocks(df_VFs_py)
print('---> blocks',vf_blocks(df_VFs_py))
print('---> missed blocks', missing_blocks(df_VFs_py))

#### now you can investigate the available information in the dataframe
#print_vf_summary(df_VFs) # this function will print

df_gi = visualFields.getgl(df_VFs) # get global indices
""" vfi, msens, ssens, tmd, tsd, pmd, psd, gh, """
""" Notice that it will calculate only missed indexes."""
df_gi.head()

############# combine them later if you need:
#combined_df = pd.concat([df_VFs, df_gi], axis=1)
#final_df = combined_df.loc[:, ~combined_df.columns.duplicated()]


---> blocks {'sens': True, 'td': True, 'pd': True, 'tdp': False, 'pdp': False}
---> missed blocks ['tdp', 'pdp']
==> py_getgl: missing global indices to compute: ['ssens', 'tmd', 'tsd', 'pmd', 'vfi']


,patientid,Gender,eyeid,FieldN,age,Time_from_Baseline,msens,MS_Cluster1,MS_Cluster2,MS_Cluster3,...,MTD_Cluster3,MTD_Cluster4,MTD_Cluster5,MTD_Cluster6,gh,ssens,tmd,tsd,pmd,vfi
0,647,F,OD,1,52.7967,0.0000,27.832885,25.57750,26.979231,30.588333,...,-3.933333,-3.730,-4.703636,-4.8725,-3.21,2.708987,-4.491743,1.475529,-1.274307,99.636326
1,647,F,OD,2,53.8234,1.0267,30.131346,27.76000,29.911538,33.263333,...,-1.200000,-1.660,-2.392727,-4.1000,-0.37,2.749958,-2.096952,1.629808,-1.701315,99.573691
2,647,F,OD,3,54.8857,2.0890,29.454808,25.47500,29.826154,32.488333,...,-1.913333,-1.805,-2.813636,-5.0150,-0.72,3.251815,-2.630071,2.045980,-1.886101,98.449774
3,647,F,OD,4,57.7331,4.9363,27.947885,23.75625,28.658462,31.431667,...,-2.805000,-2.869,-4.837273,-6.6325,-1.64,3.529175,-3.884106,2.147168,-2.204590,97.630062
4,647,F,OD,5,58.7680,5.9713,27.644038,24.90250,28.104615,30.105000,...,-4.073333,-4.515,-4.778182,-4.7975,-2.71,2.859302,-4.342051,1.838664,-1.601875,98.694345


### Get the all predefined normalization settings (from visualFields package)

In [10]:
from PyVisualFields import visualFields

NormValues = visualFields.normvals()
NormInfo = visualFields.get_info_normvals()

''' 
==> comment: > pw: pointwise, classic: smooth
==> comment: > default is: sunyiu_24d2
'''

==> comment: > pw: pointwise, classic: smooth
==> comment: > default is: sunyiu_24d2

==> SettingName:  sunyiu_24d2_pw
  name: 24-2
  perimetry: static automated perimetry
  strategy: SITA Standard
  size: Size III

==> SettingName:  sunyiu_24d2
  name: 24-2
  perimetry: static automated perimetry
  strategy: SITA Standard
  size: Size III

==> SettingName:  sunyiu_24d2_pw_cps
  name: 24-2
  perimetry: static automated perimetry
  strategy: SITA Standard
  size: Size III

==> SettingName:  sunyiu_24d2_cps
  name: 24-2
  perimetry: static automated perimetry
  strategy: SITA Standard
  size: Size III

==> SettingName:  iowa_PC26_pw
  name: Iowa pointwise NVs for PC26
  perimetry: Static automated perimetry
  strategy: ZEST
  size: Size V

==> SettingName:  iowa_PC26_pw_cps
  name: Iowa pointwise NVs for PC26 with a CPS
  perimetry: Static automated perimetry
  strategy: ZEST
  size: Size V

==> SettingName:  iowa_Peri_pw
  name: Iowa pointwise NVs for Peri
  perimetry: Static automated 

' \n==> comment: > pw: pointwise, classic: smooth\n==> comment: > default is: sunyiu_24d2\n'

### set normalization values based on predefined ones (from visualFields package)

In [11]:
from PyVisualFields import visualFields

# we are going to change the setting from classic SUNY-IU to Iowa pointwise 
print('===> current NV setting:', )
currentNV = visualFields.getnv() 
target_predeifned_nv_name = 'iowa_Peri_pw'
visualFields.setnv(target_predeifned_nv_name)
print('===> new current NV setting:', )
currentNV = visualFields.getnv() # check it is set correctly:

===> current NV setting:
$name
[1] "24-2"

$perimetry
[1] "static automated perimetry"

$strategy
[1] "SITA Standard"

$size
[1] "Size III"
===> new current NV setting:
$name
[1] "Iowa pointwise NVs for Peri"

$perimetry
[1] "Static automated perimetry"

$strategy
[1] "ZEST"

$size
[1] "Size V"


### caculate new normalization values from a population and set it to be used:

In [12]:
from PyVisualFields import visualFields
import pickle

######## caculate new nv values and set it to be used:
df_VFs_py = visualFields.data_vfctrSunyiu24d2()
newNV_r, newNV_py = visualFields.nvgenerate(df_VFs_py, method = "pointwise",
                             name = "our_NV",
                             perimetry = "something",
                             strategy = "something",
                             size = "tmp")
visualFields.setnv(newNV_r)
print('===> current NV setting: ')
currentNV = visualFields.getnv() # check it is set correctly:

#''' notcie: this normalization will not be saved.
#We need to set for each session
#so we need to save and load them seperately, e.g.: '''
newNV_dict = { "newNV_r": newNV_r, "newNV_py": newNV_py }
pickle.dump( newNV_dict, open( "our_NV.pkl", "wb" ) )

loaded_dict = pickle.load( open( "our_NV.pkl", "rb" ) )
newNV_r = loaded_dict['newNV_r']
newNV_py = loaded_dict['newNV_py']
print('===> current NV setting: ')
visualFields.setnv(newNV_r) # set it to be used
currentNV = visualFields.getnv() # check it is set correctly:


===> current NV setting: 
$name
[1] "our_NV"

$perimetry
[1] "something"

$strategy
[1] "something"

$size
[1] "tmp"
===> current NV setting: 
$name
[1] "our_NV"

$perimetry
[1] "something"

$strategy
[1] "something"

$size
[1] "tmp"


In [13]:
#### if we like to reset to default and change the normalization values to the defalut of package: sunyiu_24d2
visualFields.setdefaults()
currentNV = visualFields.getnv() # check it is set correctly:

==> default normalization setting is set: sunyiu_24d2
$name
[1] "24-2"

$perimetry
[1] "static automated perimetry"

$strategy
[1] "SITA Standard"

$size
[1] "Size III"
